# nanoGPT: a character-level Shakespeare model

**What:** trained Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT) from scratch on ~1 MB of Shakespeare, then had it write new text.
**When / where:** 1 Nov 2026, Google Colab, free **Tesla T4** GPU (16 GB). Training took ~16 minutes.
**Why:** part of `mini-agent-village`, to have actually run and read a model codebase before building the agents.

| Setting | Value |
|---|---|
| Model | GPT, 6 layers, 6 attention heads, 384-dim embeddings, **10.65M parameters** |
| Context | 256 characters |
| Data | Tiny Shakespeare: 65 distinct characters, ~1.0M train / ~0.11M val |
| Training | 5,000 steps, batch 64, learning rate 1e-3 → 1e-4, dropout 0.2 |
| Precision | float16 (the T4 has no fast bfloat16), `torch.compile` off for reliability |

## 1. Check the GPU
Should list a **Tesla T4**. If it doesn't, set *Runtime → Change runtime type → T4 GPU*.

In [1]:
!nvidia-smi

Sun Nov  1 10:49:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get the code
`%cd` (not `!cd`) so the folder change sticks for every later cell: each `!` line runs in its own throwaway shell.

In [2]:
!git clone https://github.com/karpathy/nanoGPT.git
%cd nanoGPT

Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.26 KiB | 20.44 MiB/s, done.
Resolving deltas: 100% (380/380), done.
/content/nanoGPT


## 3. Install the one missing library
Colab already has PyTorch and numpy; `sample.py` also needs `tiktoken`.

In [3]:
!pip install tiktoken

## 4. Prepare the data
Downloads Tiny Shakespeare, maps each of the 65 characters to a number, and splits it 90% train / 10% validation into `train.bin` / `val.bin`.

In [4]:
!python data/shakespeare_char/prepare.py

length of dataset in characters: 1,115,394
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


## 5. Train
Loss starts near ln(65) ≈ 4.17 (random guessing). A checkpoint is saved only when **val loss** improves (`always_save_checkpoint = False`).

In [5]:
!python train.py config/train_shakespeare_char.py --dtype=float16 --compile=False

Overriding config with config/train_shakespeare_char.py:
# train a miniature character-level shakespeare model
# good for debugging and playing on macbooks and such

out_dir = 'out-shakespeare-char'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 200
log_interval = 10 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'shakespeare-char'
wandb_run_name = 'mini-gpt'

dataset = 'shakespeare_char'
gradient_accumulation_steps = 1
batch_size = 64
block_size = 256 # context of up to 256 previous characters

# baby GPT model :)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 1e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of 

## 6. Sample
Loads the best checkpoint and writes 10 samples of 500 characters each (temperature 0.8).

In [6]:
!python sample.py --out_dir=out-shakespeare-char --dtype=float16

Overriding: out_dir = out-shakespeare-char
Overriding: dtype = float16
number of parameters: 10.65M
Loading meta from data/shakespeare_char/meta.pkl...


ANGELO:
And I will kiss you that you have done with a man
all and barnet years and very redelihood, away!

ISABELLA:
I would be strangeful. I would were long
es it enough, will it that vain and with him on that spells
of the duke he could not speak.

DUKE VINCENTIO:
How doth it?

DUKE VINCENTIO:
Not only so with mine honour, but it is not
ends of him that beheld him not.

ANGELO:
I have for thy flesh tongue Prince, Sir.

DUKE VINCENTIO:
Have done the Earl of Naples.

DUKE VINCENTIO:
Afresh thee
---------------

Menenius, I must thou and give me bear; I'll not suffer
To be reason; and being for the court.

BRUTUS:
I may stay as you would be thus wonder'd ever
A people to the stock of this poor mation: in
That the gods as you content; ask your will have not,
You have not absolute your faces: the least self your hold
In prison in the sen

## Results

| Step | Train loss | Val loss | |
|---|---|---|---|
| 0 | 4.29 | 4.28 | Random guessing |
| 500 | 1.53 | 1.73 | Learning fast |
| 1000 | 1.27 | 1.53 | |
| **1750** | **1.10** | **1.46** | **Best: the saved checkpoint** |
| 3000 | 0.86 | 1.53 | Val rising: overfitting |
| 5000 | 0.62 | 1.70 | Memorising the training text |

- **Overfitting** after step 1750: train loss kept falling while val loss rose. Saving only on val improvement acted as **early stopping**, so the sampled model is the step-1750 one.
- **The sample** learned the play format and real character names grouped by play (*Measure for Measure*, *Coriolanus*, *The Winter's Tale*), but not meaning, and it invents words (*redelihood*, *strangeful*), since it writes one character at a time.